[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/01_%E9%A1%A7%E5%AE%A2%E6%8C%87%E6%A8%99%E3%81%AE%E7%9B%B8%E9%96%A2%E5%88%86%E6%9E%90.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')
from scipy import stats

# 発展マーケ-01. 顧客指標の相関分析（相関行列・偏相関）

> Colab（ブラウザ）で実行可。最初に「① セットアップ」セルを実行。
> **統計検定2級レベル**（実務でよく使う検定・分析）。scipy・statsmodels を使用。

**舞台設定**：顧客の「購入回数」「累計売上」「最終購入からの日数(Recency)」は互いにどう関係するか。複数指標の関係を**相関行列**で一覧し、**偏相関**で「他の指標の影響を除いた純粋な関係」を見ます。

**使う統計（2級）**：相関行列・偏相関係数・見せかけの相関。

### 使うデータ：`btob_customers.csv`（顧客320件）
`購入回数`・`累計売上`・`最終購入日` から `Recency`（最終購入からの日数）を作って分析します。

## この章でできるようになること
- 複数指標の関係を **相関行列** で一覧できる
- **偏相関** で「第3の変数の影響を除いた関係」を測れる
- 相関と因果・見せかけの相関の違いを説明できる

> **前提**：相関係数（3級）　/　**所要**：約20分　/　**レベル**：2級

## 1. データを用意する

In [ ]:
cust = load('btob_customers.csv')
基準日 = pd.Timestamp('2026-01-01')
cust['Recency'] = (基準日 - pd.to_datetime(cust['最終購入日'])).dt.days
cols = ['購入回数','累計売上','Recency']
cust[cols].head()

In [ ]:
# ここに書いて試してみよう

## 2. 相関行列で全組み合わせを一覧

相関行列は、各指標どうしの相関係数をまとめた表です。+1に近いほど一緒に増え、−1に近いほど逆向きです。

In [ ]:
corr = cust[cols].corr()
print(corr.round(2))
fig, ax = plt.subplots(figsize=(4.5,4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=30, ha='right')
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)): ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center')
fig.colorbar(im); ax.set_title('相関行列'); plt.tight_layout(); plt.show()

In [ ]:
# ここに書いて試してみよう

## 3. 偏相関：他の指標の影響を除く

購入回数と累計売上は強く相関しますが、これは「よく買う人＝最近の人」など Recency が両方に効いている影響を含みます。Recency を一定にしたときの純粋な関係が **偏相関** です。

In [ ]:
def partial_corr(x, y, z, d):
    rxy, rxz, ryz = d[x].corr(d[y]), d[x].corr(d[z]), d[y].corr(d[z])
    return (rxy - rxz*ryz) / np.sqrt((1-rxz**2)*(1-ryz**2))
r  = cust['購入回数'].corr(cust['累計売上'])
pr = partial_corr('購入回数','累計売上','Recency', cust)
print(f'購入回数×累計売上  単純相関 = {r:.3f}')
print(f'　　　　　　　　　 偏相関(Recency制御) = {pr:.3f}')
print('→ Recencyの影響を除いてもほぼ高いまま＝両者は直接強く結びつく')

In [ ]:
# ここに書いて試してみよう

## まとめ
- **相関行列**で複数指標の関係をまとめて把握できる。
- **偏相関**は「第3の変数を一定にしたときの純粋な関係」。見せかけの相関を見抜くのに使う。
- 相関が高くても**因果とは限らない**。

> **よくある間違い**
> - **相関≠因果**。一緒に動いても、片方がもう片方を起こしたとは限らない。
> - 第3の変数が両方に効くと**見せかけの相関**が出る。偏相関で確認する。
> - 相関係数は**直線的な関係**しか測れない（曲線関係は低く出る）。散布図も見る。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 相関係数が取りうる最大値を ans に
ans = None   # ヒント: 相関係数の範囲は −1〜?
_check('Q1 相関係数の最大', ans, 1)

In [ ]:
import numpy as np
# Q2: 単純相関 rxy=0.8, rxz=0.5, ryz=0.5 のときの偏相関 (rxy−rxz·ryz)/√((1−rxz²)(1−ryz²)) を ans に
ans = None   # 例: (0.8-0.5*0.5)/np.sqrt((1-0.25)*(1-0.25))
_check('Q2 偏相関', ans, (0.8-0.5*0.5)/np.sqrt((1-0.25)*(1-0.25)))

In [ ]:
import numpy as np
# Q3: [1,2,3] と [2,4,6] の相関係数を ans に（完全な直線関係）
ans = None   # 例: np.corrcoef([1,2,3],[2,4,6])[0,1]
_check('Q3 相関係数', ans, np.corrcoef([1,2,3],[2,4,6])[0,1])

---
## 練習問題

**問1.** `累計売上` と `Recency` の単純相関と、購入回数を制御した偏相関を比べよう。

**問2.** 3指標の散布図行列（`pd.plotting.scatter_matrix`）を描き、関係の形を目で確かめよう。

**問3.**（考察）相関が高い2指標の例を挙げ、それが因果と言えない理由を1〜2行で書こう。

> **解答例は別ページ**（ネタバレ防止）**[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/01_%E9%A1%A7%E5%AE%A2%E6%8C%87%E6%A8%99%E3%81%AE%E7%9B%B8%E9%96%A2%E5%88%86%E6%9E%90.md)**
>
> 自分で解いてから開きましょう。

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 相関行列 | 各指標どうしの相関係数の表 |
| 偏相関 | 第3の変数を一定にした相関 |
| 見せかけの相関 | 第3の変数が生む擬似的な相関 |

```python
df[cols].corr()                       # 相関行列
# 偏相関 = (rxy − rxz·ryz)/√((1−rxz²)(1−ryz²))
```

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "06_発展_マーケ分析/01_顧客指標の相関分析"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")